In [33]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import pickle
from tqdm import tqdm
import plotly.express as px
from scipy import stats
import seaborn as sns
from utils import *
from uniqueness_utils import *
from plotting_utils import *
import json
import matplotlib
import os

In [ ]:
#SET GRAPHING PARAMETERS

#DEFINE DIRECTORY to save figures
FIGURE_DIR = 'figures'
if not os.path.isdir(FIGURE_DIR):
    os.mkdir(FIGURE_DIR)


plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 7
matplotlib.rcParams['pdf.fonttype'] = 42

from datetime import date
today = date.today()
date_str = today.strftime('%d%b%Y')
print ('Date prefix:', date_str)


plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5

In [ ]:
# Load biosynthesis graph
## CURRENTLY REQUIRES MANUALLY COPYING THE RELEVANT FILES TO DATA_DIR
DATA_DIR = '00_data/'
GRAPH_PATH = '../01_biochemical_network/00_data/processed/graph/28Jan2025_whole_metabolic_network_labeled.pkl'

with open(GRAPH_PATH, 'rb') as f:
    met = pickle.load(f)


print (len([n for n in met.nodes if '>' not in n]))    
print (len([n for n in met.nodes if '>'  in n]))    

# Include protein sequence knowledge
reaction_nodes = np.unique([n for n in met.nodes if '>'  in n])
for n in reaction_nodes:
    if pd.isna(met.nodes[n]['aa_seq_len']): #or met.nodes[n]['aa_seq_len']>1000:
        met.remove_node(n)

print (len([n for n in met.nodes if '>' not in n]))    
print (len([n for n in met.nodes if '>'  in n]))           

# Get names to smiles mapping
n_t_s_df = pd.read_csv(DATA_DIR+'17Jul2023_name-smiles-correspondence.csv')
n_t_s_df['cleaned_name'] = n_t_s_df['name'].map(lambda x : clean_name(str(x)))

In [ ]:
# Get steps counts for different organisms
organism_abbrev = 'EC'
building_blocks = pd.read_csv(DATA_DIR+'e_coli_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values) if '*' not in s] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))


organism_abbrev = 'BS'
building_blocks = pd.read_csv(DATA_DIR+'b_subtilis_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values)] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))

# Redefine buyables and step count for E Coli
organism_abbrev = 'AT'
building_blocks = pd.read_csv(DATA_DIR+'a_thaliana_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values) if '*' not in s] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))


# Redefine buyables and step count for E Coli
organism_abbrev = 'RG'
building_blocks = pd.read_csv(DATA_DIR+'r_gelatinosus_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values) if '*' not in s] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))

organism_abbrev = 'HS'
building_blocks = pd.read_csv(DATA_DIR+'h_sapiens_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values)] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))

organism_abbrev = 'AN'
building_blocks = pd.read_csv(DATA_DIR+'a_nidulans_metabolites_from_metanetx_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values)] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))


organism_abbrev = 'PP'
building_blocks = pd.read_csv(DATA_DIR+'p_putida_metabolites_from_pathways.csv', sep='\t')

std_building_blocks = set([standardize_smiles(str(s)) for s in tqdm(building_blocks['smiles'].values)] + ['[Mg+2]'])

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))

In [6]:
#Load predicted spectra

pred_spec = pd.read_csv(DATA_DIR+'16Jul2023_all_done_spectra.csv')
pred_spec.columns = [x.replace('_b3lyp','').split('/')[-1] for x in pred_spec.columns] 
pred_spec = pred_spec.loc[:, ~pred_spec.columns.duplicated()]
index = pred_spec['Unnamed: 0']
pred_spec = pred_spec.drop(columns=['Unnamed: 0'])
pred_spec.index = index

In [ ]:
# load precomputed 

uniqueness_scores = pd.read_csv(DATA_DIR+'05Jun2024_consolidated_uniqueness_scores.csv', sep='\t')
print (len(uniqueness_scores))
uniqueness_scores = uniqueness_scores.dropna(subset=['name'])
print (len(uniqueness_scores))

In [ ]:
uniqueness_scores.head(10)

In [9]:
uniqueness_scores['max_peak'] = [pred_spec.loc[:,n].max() for n in uniqueness_scores['name']] #min_max_norm([pred_spec.loc[:,n].max() for n in uniqueness_scores['name']])
uniqueness_scores['summed'] = min_max_norm([pred_spec.loc[:,n].sum() for n in uniqueness_scores['name']])
uniqueness_scores['auc'] = min_max_norm([np.trapz(pred_spec.loc[:,n]) for n in uniqueness_scores['name']])

In [ ]:
top_scorers = uniqueness_scores[uniqueness_scores['lambda_max']>100].sort_values(by='ws_dist', ascending=False).head(100)
print (top_scorers['name'].tolist())
print (top_scorers['smiles'].tolist())
top_scorers

top_scorers = uniqueness_scores[uniqueness_scores['lambda_max']>100].sort_values(by='ws_dist', ascending=False).head(10)

## TOP HITS NO SYNTHETIC CONSIDERATION

In [ ]:
for idx in top_scorers.index:
    print (top_scorers.loc[idx,'name'])
    print (top_scorers.loc[idx,'smiles'])
    print (top_scorers.loc[idx, 'ws_dist'])
    print ('AUC:', np.trapz(pred_spec[top_scorers.loc[idx,'name']]))
    print(pred_spec[top_scorers.loc[idx,'name']].max()*30)
    smiles = top_scorers.loc[idx, 'smiles']
    plt.figure(figsize=(1,0.7))
    plt.plot(pred_spec[top_scorers.loc[idx,'name']], color='cornflowerblue')
    plt.xlim(100,1000)
    plt.minorticks_on()
    plt.yticks([])
    plt.xticks([100,500,900])
    plt.tight_layout()
    
    plt.show()

In [ ]:
plt.hist(pred_spec.max(axis=0).tolist(),bins=100)
plt.ylabel('Maximal absorbance intensity')
plt.show()

### ADD SYNTHETIC CONSIDERATION

In [ ]:
REACTION_DATA_FILE = DATA_DIR+'21Nov2023_all_reaction_from_rhea_bkms_metacyc.csv'

reaction_w_rev_df = pd.read_csv(REACTION_DATA_FILE, sep='\t')
for c in ['level_0', 'Unnamed: 0.1', 'Unnamed: 0']:
    if c in reaction_w_rev_df.columns:
        reaction_w_rev_df = reaction_w_rev_df.drop(columns=[c])

In [14]:
uniqueness_scores.loc[:,'norm_ws_dist'] = min_max_norm(uniqueness_scores['ws_dist'])

In [ ]:
uniqueness_scores.sort_values('ws_dist', ascending=False).head(10)

In [ ]:
# correct error in RG metabolism annotation

bchl = standardize_smiles(r'CC[C@@H]1([C@@H](C)C2(\N=C1\C=C7(C(\C)=C3(C4(\N([Mg]N6(C(\C=2)=C(C(/C)=C(\C=C5([C@@H](C)[C@H](CCC(=O)OC/C=C(C)/CCC[C@H](C)CCC[C@H](C)CCCC(C)C)C(\C(/[C@@H-](C(OC)=O)C(=O)3)=4)=N5))/6)/C(C)=O))7)))))')
bchl_precursors = set([bchl])
bchl_path = met.nodes[bchl]['shortest_path_RG']
for i,step in enumerate(bchl_path):
    reactant = [x for x in step.split('>>')[0].split('.') if 'Mg' in x][0]
    product = [x for x in step.split('>>')[1].split('.') if 'Mg' in x][0]
    bchl_precursors.add(reactant)
    bchl_precursors.add(product)

    
    
organism_abbrev = 'RG_adj'
std_building_blocks = set([n for n in met.nodes if met.nodes[n]['path_length_RG']==0]).union(bchl_precursors)

starting_nodes = []
for node in met.nodes:
    if node in std_building_blocks:
        starting_nodes.append(node)

met = metabolic_dijkstra(met, starting_nodes, path_length_field="path_length_{}".format(organism_abbrev),
                         shortest_path_field="shortest_path_{}".format(organism_abbrev))
uniqueness_scores['RG_adj_steps'] = [met.nodes[s]['path_length_RG_adj'] if s in met.nodes else np.inf for s in uniqueness_scores['smiles'] ]

In [ ]:
uniqueness_scores['mols'] = [Chem.MolFromSmiles(x) for x in uniqueness_scores['std_smiles']]
uniqueness_scores['fps'] = [AllChem.GetMorganFingerprintAsBitVect(x, nBits=2048, radius=2) for x in uniqueness_scores['mols']]
x_metric = 'norm_ws_dist'
tan_thresh = 0.45
highlighted_ls = ['biliverdin_ixa', '0x38f0bd976d71017']
to_plot = uniqueness_scores.copy()

sim_highlighted = {n:[] for n in highlighted_ls}

for name in highlighted_ls:
    entry1 = uniqueness_scores[uniqueness_scores['name']==name]
    try:
        fp = to_plot[to_plot['name'] == name]['fps'].values[0]
        to_plot['sim_to_{}'.format(name)] = DataStructs.BulkTanimotoSimilarity(fp, to_plot['fps'].values)
        sim_names = to_plot[(to_plot['sim_to_{}'.format(name)]>tan_thresh) & (to_plot['sim_to_{}'.format(name)]<1)]['name']
        for n in sim_names:
            entry2 = to_plot[to_plot['name']==n]
            if entry2[x_metric].values[0] > entry1[x_metric].values[0] :
                sim_highlighted[name].append(n)
    except IndexError:
        pass

In [ ]:
sim_highlighted

In [ ]:
to_plot[to_plot[f'sim_to_{highlighted_ls[0]}']>0.45]

In [ ]:
ext_coeff =  30*uniqueness_scores['max_peak']
uniqueness_scores['max_ext_coeff'] = ext_coeff
uniqueness_scores.loc[ext_coeff>.1, :].sort_values('max_peak')['max_peak']

In [ ]:
uniqueness_scores['min_steps'] = np.min(uniqueness_scores[[c for c in uniqueness_scores.columns if 'steps' in c]], axis=1)
carot = Chem.MolFromSmarts('C=CC(C)=CC=CC(C)=CC=CC=C(C)C=CC=C')
porph = Chem.MolFromSmarts('[#6]~1~[#6]~[#6]~[#7]~[#6]~[#6]~[#6]~[#7]~[#6]~[#6]~[#6]~[#7]~[#6]~[#6]~[#6]~[#7]~1')
for idx in tqdm(uniqueness_scores.index):
    if uniqueness_scores.loc[idx, 'mols'].HasSubstructMatch(carot) or uniqueness_scores.loc[idx, 'mols'].HasSubstructMatch(porph):
        uniqueness_scores.loc[idx, 'min_steps'] = np.inf

In [ ]:
# import importlib
# importlib.reload(spectranalysis.utils)
# from spectranalysis.uniqueness_utils import *
# from spectranalysis.utils import *


to_highlight=[]
kwargs = {'metric':'ws_dist', 'max_steps':0, 'num_top_cands':30}

bb_set_name = 'e-coli'
organism = r'$\it{E. coli}$'
organism_abbrev = 'EC'
print (organism)


organism_info = {}

organism_info['EC'] = {'bb_set_name':'e-coli', 'organism':r'$\it{E. coli}$'}
organism_info['RG_adj'] = {'bb_set_name':'r_gelatinosus', 'organism':r'$\it{R. gelatinosus}$'}
organism_info['HS'] = {'bb_set_name':'h_sapiens', 'organism':r'$\it{H. sapiens}$'}
organism_info['AT'] = {'bb_set_name':'a_thaliana', 'organism':r'$\it{A. thaliana}$'}
organism_info['AN'] = {'bb_set_name':'a_nidulans', 'organism':r'$\it{A. nidulans}$'}
organism_info['PP'] = {'bb_set_name':'p_putida', 'organism':r'$\it{P. putida}$'}
organism_info['PR'] = {'bb_set_name':'p_rubens', 'organism':r'$\it{P. rubens}$'}
organism_info['min'] = {'bb_set_name':'min', 'organism':'Min'}


top_cands = highlighted_ls
secondary_highlights = []
for v in sim_highlighted.values():
    secondary_highlights += v

greens =  plt.get_cmap('tab20')
    
top_cands_colors = {}
cmap = plt.get_cmap('tab20')


for n in sim_highlighted['biliverdin_ixa']:
    top_cands_colors[n] = '#dcb6de'
top_cands_colors['biliverdin_ixa'] = '#D8529E'

for n in sim_highlighted['0x38f0bd976d71017']:
    top_cands_colors[n] = cmap(1)    
top_cands_colors['0x38f0bd976d71017'] = cmap(0)

print (top_cands)
print (isinstance(top_cands_colors, dict))
for organism_abbrev in organism_info.keys():
    print (organism_abbrev)
    bb_set_name = organism_info[organism_abbrev]['bb_set_name']
    organism = organism_info[organism_abbrev]['organism']
    
    
    plot_steps_vs_uniqueness_for_organism(uniqueness_scores.loc[uniqueness_scores['max_ext_coeff']>0.08, :], bb_set_name, organism, organism_abbrev, top_cands,
                                          secondary_names_to_highlight=secondary_highlights, #+ top_cands,
                                          secondary_color=top_cands_colors, secondary_size = 6, fig_size=(1.38,1.38),
                                          **kwargs)

In [ ]:
bchl_score = uniqueness_scores[uniqueness_scores['smiles']==bchl]['norm_ws_dist'].values[0]
Chem.Draw.MolsToGridImage(uniqueness_scores[(uniqueness_scores['RG_adj_steps']==0) & (uniqueness_scores['norm_ws_dist']>bchl_score)]['mols'].to_list(),
                         legends=uniqueness_scores[(uniqueness_scores['RG_adj_steps']==0) & (uniqueness_scores['norm_ws_dist']>bchl_score)]['name'].to_list())

In [ ]:
# Browse interactively
import plotly.express as px

to_show = uniqueness_scores.copy()
to_show['AT_steps'] = [x if x < 11 else 14 for x in uniqueness_scores['AT_steps']]
fig = px.scatter(to_show, x='norm_ws_dist', y='AT_steps', hover_data=['name'])
fig.show()

In [ ]:
## Look at synthesis pathways for candidate reporters

candidates = uniqueness_scores[uniqueness_scores['name'].map(lambda x : x in ['biliverdin_ixa', '152362311996990386169628651672288219332'])]

for idx in candidates.index:
    print ('\n\n\n')
    print (candidates.loc[idx,'name'])
    print (candidates.loc[idx,'AT_steps'])
    print (candidates.loc[idx, 'norm_ws_dist'])
    

    
    smiles = candidates.loc[idx, 'smiles']
    show_rxn_list(met.nodes[smiles]['shortest_path_EC'], reaction_info_df = reaction_w_rev_df)
    plt.figure(figsize=(1.5,1))
    plt.plot(pred_spec[candidates.loc[idx,'name']], color='purple')
    plt.xlim(350,1050)
    plt.tight_layout()
    print (candidates.loc[idx,'name'])
    plt.show()

In [ ]:
candidates = uniqueness_scores[uniqueness_scores['name'].map(lambda x : x in ['0x38f0bd976d71017'])]

for idx in candidates.index:
    print (candidates.loc[idx,'name'])
    print (candidates.loc[idx,'AT_steps'])
    print (candidates.loc[idx, 'norm_ws_dist'])
    

    
    smiles = candidates.loc[idx, 'smiles']
    show_rxn_list(met.nodes[smiles]['shortest_path_AT'], reaction_info_df = reaction_w_rev_df)
    plt.figure(figsize=(1.5,1))
    plt.plot(pred_spec[candidates.loc[idx,'name']], color='purple')
    plt.xlim(350,1050)
    plt.tight_layout()
#     plt.savefig('{}/predicted_spectra/{}_predicted_spectrum.pdf'.format(FIGURE_DIR, candidates.loc[idx,'name']),dpi=400)
    plt.show()

## Compare predicted and experimental spectra

In [34]:

with open('00_data/experimental_data/spectral_responses_reflectance.json', 'r') as f:
    experimental_spectra = json.load(f)


spectrophotometer_measurements = pd.read_csv('00_data/experimental_data/spectrophotometer_measuremenets.csv')
key = {1:10e-3, 
       2:5e-3, 
       3:2.5e-3,
       4:1e-3,
       5:5e-4,
       6:2.5e-4,
       7:1e-4,
       8:5e-5,
       9:2.5e-5,
       10:1e-5,
       'A':'hemin',
       'B':'bacteriochlorophyll',
       'C':'biliverdine',
       'D':'melanin',
       'E':'naringenin',
       'F':'quercetin',
       'G':'protoporphyrin',
       'H':'beta-carotene',
       'I':'DMSO'
      }
spectrophotometer_measurements.index = spectrophotometer_measurements['Wavelength']

In [ ]:
for c in spectrophotometer_measurements.filter(regex='B.*').columns:
    print (c)
    plt.figure()
    plt.plot(spectrophotometer_measurements['Wavelength'],  spectrophotometer_measurements[c])
    plt.show()
    
for c in spectrophotometer_measurements.filter(regex='C.*').columns:
    print (c)
    plt.figure()
    plt.plot(spectrophotometer_measurements['Wavelength'],  spectrophotometer_measurements[c])
    plt.show()

In [ ]:
plt.figure(figsize=(1.5,1))
plt.plot(pred_spec['biliverdin_ixa'].index, pred_spec['biliverdin_ixa']/np.max(pred_spec['biliverdin_ixa']), 
#          color=(236/255,0,140/255,1))
         color = 'orange')

plt.plot(spectrophotometer_measurements['Wavelength'],spectrophotometer_measurements['C6']/spectrophotometer_measurements.loc[:,'C6'].max(), color='black')
plt.xlim(300,1000)
plt.xticks([400,600,800,1000])
plt.minorticks_on()
plt.tick_params(axis='y', which='minor', left=False)
plt.savefig('{}/compare_experimental_uvvis_and_predicted_biliverdinIXa_in_cells.pdf'.format(FIGURE_DIR))
plt.show()

In [ ]:
plt.figure(figsize=(1.5,1))
plt.plot(pred_spec.index, pred_spec['0x38f0bd976d71017']/np.max(pred_spec['0x38f0bd976d71017']),
#          color=(236/255,0,140/255,1))
         color = 'orange')
plt.plot(spectrophotometer_measurements['Wavelength'],min_max_norm(spectrophotometer_measurements['B1']), color='black')

plt.xlim(300,1000)
plt.xticks([400,600,800,1000])
plt.minorticks_on()
plt.tick_params(axis='y', which='minor', left=False)
plt.savefig('{}/compare_experimental_uvvis_and_predicted_bchla.pdf'.format(FIGURE_DIR))
plt.show()

In [ ]:
bg_spec = np.array(experimental_spectra['YF10-concentrations_map_frontside']['0.0'])
target_spec = np.array(experimental_spectra['YF10-concentrations_map_frontside']['400.0'])
wls = np.load('00_data/background_spectra/wavelengths_from_ecoli_on_agar_experiment.npy')
plt.figure(figsize=(1.5,1.5))

reconstructed_spec = (bg_spec-target_spec) / bg_spec
plt.plot(wls, reconstructed_spec/np.max(reconstructed_spec), color='black')


plt.xlim(400,1000)
plt.xticks([400,600,800,1000])
plt.yticks([0,0.5,1])
plt.minorticks_on()
plt.tick_params(axis='y', which='minor', left=False)
plt.savefig('{}/infered_bchla_absorbance.pdf'.format(FIGURE_DIR))
plt.show()

In [ ]:
bg_spec = np.array(experimental_spectra['bphO-smurfp_concentrations_map_frontside']['0.0'])
target_spec = np.array(experimental_spectra['bphO-smurfp_concentrations_map_frontside']['5000.0'])
plt.figure(figsize=(1.5,1.5))

reconstructed_spec = (bg_spec-target_spec) / bg_spec
plt.plot(wls, reconstructed_spec/np.max(reconstructed_spec), color='black')


plt.xlim(400,1000)
plt.xticks([400,600,800,1000])
plt.yticks([0,0.5,1])
plt.minorticks_on()
plt.tick_params(axis='y', which='minor', left=False)
plt.savefig('{}/infered_blvda_absorbance.pdf'.format(FIGURE_DIR))
plt.show()

In [ ]:
plt.figure(figsize=(1.5,1.5))

# pure molecule
plt.plot(spectrophotometer_measurements['Wavelength'],spectrophotometer_measurements['C6']/spectrophotometer_measurements.loc[:300,'C6'].max(), color='black')
plt.xlim(400,1000)
plt.ylim(-.1,1)
plt.minorticks_on()
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance (norm)')
plt.savefig(f'{FIGURE_DIR}/pure_blvda_spectrophotomer_spectrum.pdf')
plt.show()

plt.figure(figsize=(1.5,1.5))
# cells
plt.plot(wls, reconstructed_spec/np.max(reconstructed_spec), color='black')
plt.minorticks_on()
plt.xlim(400,1000)
plt.ylim(-.1,1)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance (norm)')
plt.savefig(f'{FIGURE_DIR}/infered_blvda_smurfp_hsi_spectrum.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(1.5,1.5))

# pure molecule
plt.plot(spectrophotometer_measurements['Wavelength'],spectrophotometer_measurements['B2']/spectrophotometer_measurements.loc[:400,'B2'].max(), color='black')
plt.xlim(400,1000)
plt.ylim(-.1,1)
plt.minorticks_on()
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance (norm)')
plt.savefig(f'{FIGURE_DIR}/pure_bchla_spectrophotomer_spectrum.pdf')
plt.show()


bg_spec = np.array(experimental_spectra['YF10-concentrations_map_frontside']['0.0'])
target_spec = np.array(experimental_spectra['YF10-concentrations_map_frontside']['10000.0'])
plt.figure(figsize=(1.5,1.5))

reconstructed_spec = (bg_spec-target_spec) / bg_spec
# plt.plot(wls, reconstructed_spec/np.max(reconstructed_spec), color='black')


plt.figure(figsize=(1.5,1.5))
# cells
plt.plot(wls, reconstructed_spec/np.max(reconstructed_spec), color='black')
plt.minorticks_on()
plt.xlim(400,1000)
plt.ylim(-.1,1)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance (norm)')
plt.savefig(f'{FIGURE_DIR}/infered_bchl_yf6_hsi_spectrum.pdf')
plt.show()

## Background comparison

In [50]:
sand_diff_scores = np.load(f'{DATA_DIR}/background_spectra/2Apr2024_sand_calculated_distance_to_diffs.npy',allow_pickle=True)
soil_diff_scores = np.load(f'{DATA_DIR}/background_spectra/2Apr2024_soil_calculated_distance_to_diffs.npy',allow_pickle=True)
grass_diff_scores = np.load(f'{DATA_DIR}/background_spectra/2Apr2024_grass_calculated_distance_to_diffs.npy',allow_pickle=True)

arm_IL_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_arm_IL_contrast.npy',allow_pickle=True)
arm_AF_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_arm_AF_contrast.npy',allow_pickle=True)
arm_W_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_arm_W_contrast.npy',allow_pickle=True)
lb_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_lb_contrast.npy',allow_pickle=True)
fabrics_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_fabric_contrast.npy',allow_pickle=True)
extinguisher_contrast_scores = np.load(f'{DATA_DIR}/background_spectra/04Jul2024_extinguisher_contrast.npy',allow_pickle=True)


In [51]:
diff_df = pd.DataFrame({'name':sand_diff_scores[0], 
                        'sand_diff_dist':sand_diff_scores[1], 
                        'soil_diff_dist':soil_diff_scores[1],
                        'grass_diff_dist':grass_diff_scores[1],
                        
                       })
lboz_df = pd.DataFrame({'name':arm_IL_contrast_scores[0], 
                        'arm_IL_diff_dist':arm_IL_contrast_scores[1],
                        'arm_AF_diff_dist':arm_AF_contrast_scores[1],
                        'arm_W_diff_dist':arm_W_contrast_scores[1],
                       'lb_diff_dist':lb_contrast_scores[1],
                       'fabrics_diff_dist':fabrics_contrast_scores[1],
                       'extinguisher_diff_dist':extinguisher_contrast_scores[1],}
                      )

In [52]:
for c in diff_df.columns:
    if c != 'name':
        if c not in uniqueness_scores.columns:
            try:
                uniqueness_scores = uniqueness_scores.merge(diff_df.loc[:,['name', c]], how='left', on='name')
            except:
                print ('Did not merge', c)

        else:
            try:
                uniqueness_scores = uniqueness_scores.drop(columns=[c])
                uniqueness_scores = uniqueness_scores.merge(diff_df.loc[:,['name', c]], how='left', on='name')
                print ('Remerged', c)
            except:
                print ('Did not merge', c)
                
for c in lboz_df.columns:
    if c != 'name':
        if c not in uniqueness_scores.columns:
            try:
                uniqueness_scores = uniqueness_scores.merge(lboz_df.loc[:,['name', c]], how='left', on='name')
            except Exception as e:
                print ('Did not merge', c, e)

        else:
            try:
                uniqueness_scores = uniqueness_scores.drop(columns=[c])
                uniqueness_scores = uniqueness_scores.merge(lboz_df.loc[:,['name', c]], how='left', on='name')
                print ('Remerged', c)
            except:
                print ('Did not merge', c)

In [ ]:
for c in uniqueness_scores.columns:
    if c[-9:] == 'diff_dist' and c[:4]!='norm':
        print (c)
        uniqueness_scores['norm_'+c] = min_max_norm(uniqueness_scores[c])

In [ ]:
plt.rcParams['font.size'] = 7

metrics = ['sand_diff_dist', 
           'soil_diff_dist',
           'grass_diff_dist',
           
           'arm_IL_diff_dist',
            'arm_AF_diff_dist',
            'arm_W_diff_dist',
           'lb_diff_dist',
           'fabrics_diff_dist',
           'extinguisher_diff_dist',
           
          ]
names = ['Sand', 
         'Soil',
         'Grass',
         
         'Arm 1',
         'Arm 2',
         'Arm 3',
         'LB Agar',
         'Dyed fabrics',
         'Red metal'
         
        ]
metrics_and_names = zip(metrics, names)
cmap = plt.get_cmap('tab20')
for y_metric, bg_name in metrics_and_names:
    print (y_metric, bg_name)
    x_metric = 'ws_dist'
    tan_thresh = 0.45
    highlighted_big = {'biliverdin_ixalpha':3, 'biliverdin_ixa':3, '0x38f0bd976d71017':4}
    highlighted = {
#         'biliverdin_ixalpha':3, 
        'biliverdin_ixa':3,'0x38f0bd976d71017':4}
    to_plot = uniqueness_scores.loc[uniqueness_scores['max_ext_coeff']>0.1, :].copy()
    
    for name in highlighted_big.keys():
        entry1 = to_plot[to_plot['name']==name]
        try:
            fp = to_plot[to_plot['name'] == name]['fps'].values[0]
            to_plot['sim_to_{}'.format(name)] = DataStructs.BulkTanimotoSimilarity(fp, to_plot['fps'].values)
            sim_names = to_plot[(to_plot['sim_to_{}'.format(name)]>tan_thresh) & (to_plot['sim_to_{}'.format(name)]<1)]['name']
            for n in sim_names:
                entry2 = to_plot[to_plot['name']==n]
                if entry2[x_metric].values[0] > entry1[x_metric].values[0] :#and entry2[y_metric].values[0] > entry1[y_metric].values[0]:
                    highlighted[n] = highlighted_big[name]-2
        except IndexError:
            pass

    color = np.array([0 if x not in highlighted.keys() else highlighted[x] for x in to_plot['name']])
    to_plot['color'] = color
    to_plot = to_plot.sort_values(by='color')
    size = np.array([0 if x not in highlighted_big.keys() else 1 for x in to_plot['name']])
    linewidth = np.array([0.1 if x not in highlighted.keys() else 0.4 for x in to_plot['name']])
    
    to_plot['size'] = size
    
    cmap = plt.get_cmap('tab20')

    palette ={0:'darkgray', 
              3:'#D8529E', 
              4:cmap(0), 
              1:'#dcb6de', 
              2:cmap(1)}#, cmap(4), cmap(5)]
    plt.figure(figsize=(1.26,1.26), dpi=400)
    sns.scatterplot(to_plot, x=x_metric, y = y_metric,
                  s = 5, legend=False,
                    #space=0,dropna=True, ylim=[-0.05,1.05], xlim=[-0.05,1.05], height=1.5 #JOINT PLOT PARAMS
                  linewidth=linewidth,
                    edgecolor='black',
                  color='darkgray', hue='color', size=to_plot['color'], 
                sizes={0:0.6,1:6,2:6, 3:12, 4:12},
                palette = palette)
    plt.xlabel('Uniqueness')
    plt.ylabel('Contrast to {}'.format(bg_name))
    ylim_max = max([0.6, 1.05*np.max(to_plot[y_metric])])
    plt.ylim(0,ylim_max)
    plt.xticks([200,400])
#     plt.ylim(-0.05,1.05)
#     plt.xticks([0,0.5,1])
#     plt.yticks([0,0.5,1])
    plt.minorticks_on()
                  #)#ratio=0.3, 
    #               hue=['black']*len(pred_scores_lboz[1]),palette=['black'])
    plt.tick_params(axis='both', which='minor', right=False, top=False)
    plt.savefig('{}/{}_{}-vs-{}_jointplots_all_metabs.pdf'.format(FIGURE_DIR, date_str,x_metric,y_metric))
#     plt.tight_layout()
#     plt.savefig('background_spectra/{}_{}-vs-{}_jointplots.png'.format(date_str,x_metric,y_metric), dpi=400)
    plt.show()

## Visualize Wasserstein

In [ ]:
import matplotlib.pylab as pl
import ot
import ot.plot
from ot.datasets import make_1D_gauss as gauss

def resample_spectrum(spectrum_wavelengths, spectrum_intensities, target_wavelengths):
    df = pd.DataFrame({'wavelengths':spectrum_wavelengths, 'int':spectrum_intensities})
    df = df.merge(pd.DataFrame({'wavelengths':target_wavelengths}), how='outer', on='wavelengths')

    df = df.sort_values(by='wavelengths')
    df.index = df['wavelengths']

    df = df.interpolate(method='index')
    df = df.loc[target_wavelengths, :]

    return df['int'].values

In [ ]:
from matplotlib import gridspec

def plot1D_mat(xs, a, b, M, title='', cmap='Reds', full_xs=None, full_a=None, full_b=None):
    r""" Plot matrix :math:`\mathbf{M}`  with the source and target 1D distribution

    Creates a subplot with the source distribution :math:`\mathbf{a}` on the left and
    target distribution :math:`\mathbf{b}` on the top. The matrix :math:`\mathbf{M}` is shown in between.


    Parameters
    ----------
    a : ndarray, shape (na,)
        Source distribution
    b : ndarray, shape (nb,)
        Target distribution
    M : ndarray, shape (na, nb)
        Matrix to plot
    """
    na, nb = M.shape
    gs = gridspec.GridSpec(4, 4)

    ax1 = pl.subplot(gs[0, 1:])
    ax2 = pl.subplot(gs[1:, 0])
    ax3 = pl.subplot(gs[1:, 1:], sharex=ax1, sharey=ax2)

    reduced_idxs = np.array(range(len(xs)))%2==0
    ax1.plot(full_xs, full_b, label='Target distribution', color='k', linewidth=0.5)
    ax1.bar(xs, b, color=(236/255,0,140/255,1),width=xs[1]-xs[0], edgecolor='black', linewidth=0.0, alpha=0.4)
    ax1.set_yticks([])
    ax1.set_title(title)
    ax1.set_xticks([])
    ax1.set_yticks([])
    

    
#     ax2.plot(a, xs, label='Source distribution', color='Gray')
    ax2.plot(full_a, full_xs, label='Source distribution', color='k', linewidth=0.5)
    ax2.barh(xs, a, color='Gray', height=xs[1]-xs[0], edgecolor='black', linewidth=0.0, alpha=0.4)
    
    ax2.invert_xaxis()
    ax2.invert_yaxis()
    ax2.set_ylim(xs[0],xs[-1])
    
    ax2.set_xticks([])
    ax2.set_yticks([])
    

    
    im = ax3.imshow(M, interpolation='nearest', cmap=cmap, extent=(xs[0], xs[-1],xs[-1],xs[0]), origin='upper')

    ax3.set_xticks([])
    pl.tight_layout()
    
    pl.subplots_adjust(wspace=0., hspace=0)
    ax3.set_xticks([200,400,600,800,1000])
    ax3.set_yticks([200,400,600,800,1000])
    
    for a in [ax1, ax2, ax3]:
        a.label_outer()
    pl.tick_params(axis='both',which='both', right=True, left=False, top=False, bottom=True)

xs = np.linspace(150,1050,20)

a_spec_full = pred_spec['biliverdin_IXalpha']
a_spec = resample_spectrum(a_spec_full.index, a_spec_full.values, xs)
a_spec = a_spec / np.trapz(a_spec)

b_spec_full = pred_spec['cobiinamide']
b_spec = resample_spectrum(b_spec_full.index, b_spec_full.values, xs)
b_spec = b_spec / np.trapz(b_spec)

# use fast 1D solver
G0, log = ot.emd_1d(xs, xs, a_spec, b_spec, dense=True, log=True,)


plt.figure(figsize=(1.85,1.85), dpi=200)
# G0[G0==0]=np.nan
G0 = G0**.6
plot1D_mat(xs, a_spec, b_spec, G0,'', 'Reds', b_spec_full.index, 
           a_spec_full.values/np.max(a_spec_full.values)*np.max(a_spec),  
           b_spec_full.values/np.max(b_spec_full.values)*np.max(b_spec))
plt.savefig(f'{FIGURE_DIR}/{date_str}_optimal_transport_example.pdf')
plt.show()

## Correlates of uniqueness

In [ ]:
# all_uniqueness_scores = pd.read_csv('data/27Jan2024_LBOZ_consolidated_uniqueness_scores.csv', sep='\t')
all_uniqueness_scores = pd.read_csv('00_data/05Jun2024_consolidated_uniqueness_scores.csv', sep='\t')
print (len(all_uniqueness_scores))
all_uniqueness_scores = all_uniqueness_scores.dropna(subset=['name'])

In [ ]:
peaks_info = []
for c in pred_spec.columns:
    spec = pred_spec[c].values
    peak_info = list(get_peaks(spec/np.max(spec), pred_spec.index,  prominence = .1, width = 5))
    peaks_info.append(peak_info)
    
peak_info_df = pd.DataFrame({'name':pred_spec.columns, 'peak_info':peaks_info})
peak_info_df['num_peaks'] = peak_info_df['peak_info'].map(lambda x : len(x))

In [86]:
import ast
# force reading as list
try:
  all_uniqueness_scores['num_peaks'] = all_uniqueness_scores['peak_info'].map(lambda x : len(ast.literal_eval(x)))
except:
  all_uniqueness_scores['num_peaks'] = all_uniqueness_scores['peak_info'].map(lambda x : len(x))
all_uniqueness_scores = all_uniqueness_scores.sort_values('ws_dist', ascending=False)

In [ ]:
print (len(peak_info_df[peak_info_df['num_peaks']>0]))
plt.figure(figsize=(1.5,1.3))

sns.countplot(peak_info_df[peak_info_df['num_peaks']>0], width=0.5, x='num_peaks', log=True, color='black')

plt.ylim(8,3e4)
plt.xlabel('Absorbance peaks (#)')
plt.ylabel('Metabolites (#)')

plt.savefig(f'{FIGURE_DIR}/{date_str}_number_of_peaks_distribution.pdf', dpi=300)

In [ ]:
for peak_num in peak_info_df['num_peaks'].drop_duplicates():
    print ('Fraction of dataset with {} peaks: {:.2f}% ({} total)'.format(peak_num, 100*np.sum(peak_info_df['num_peaks']==peak_num) / len(peak_info_df['num_peaks']), np.sum(peak_info_df['num_peaks']==peak_num) ))

In [ ]:
sum(all_uniqueness_scores['num_peaks']==3)

In [ ]:
print('.'.join(all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['smiles'].tolist()))

In [ ]:
Chem.Draw.MolsToGridImage([Chem.MolFromSmiles(x) for x in all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['smiles']],
                          legends=[x for x in all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['name']],
                          subImgSize=(300,300))

In [ ]:
print( '\n'.join([str(i+1)+'.'+str(x) for i, x in enumerate(all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['name'])]))

In [ ]:
print( '\n'.join([str(i+1)+'.'+str(x) for i, x in enumerate(all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['smiles'])]))

In [ ]:
print ('\n'.join([x for x in all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['smiles']]))

In [ ]:
all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['name'].tolist()

In [ ]:
for n in all_uniqueness_scores[all_uniqueness_scores['num_peaks']==3]['name'].tolist():
    plt.figure(figsize=(1.1,0.9))
    plt.plot(pred_spec.loc[:, n], color='black')
    if n!='9R_10S_12S_13S_14R_16S_18R-13-ethyl-8-methyl-8_15-diazahexacyclo14_2_1_01_9_02_7_010_15_012_17nonadeca-2_4_6-triene-14_18-diol':
        plt.xticks([250,500,750,1000])
        plt.xlim(100,1150)
    else:
        plt.xlim(100,1450)
        plt.xticks([250,750,1250])
    plt.ylabel('Absorbance')
    print (f'{FIGURE_DIR}/predicted_spectra/3_peaks_{date_str}_{n}_predicted_spectrum.pdf')
    plt.savefig(f'{FIGURE_DIR}/predicted_spectra/3_peaks_{date_str}_{n}_predicted_spectrum.pdf')
    plt.show()

In [ ]:
plt.figure(figsize=(2.5,2))
plt.hist(all_uniqueness_scores[all_uniqueness_scores['num_peaks']==1]['lambda_max'].values, 
         bins=40, range=(100,900), log=True, color='black')
plt.ylabel('Number of molecules')
plt.xlabel('Wavelength (nm)')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/distribution_of_single_peak_lmax.png', dpi=300)
plt.show()


In [97]:
all_uniqueness_scores['norm_ws_dist'] = min_max_norm(all_uniqueness_scores['ws_dist'])

In [68]:
def plot_outline (x, y ,step_size, color="black", linewidth=0.5):
    ys = []
    xs = []
    xs.append(x[0] - 0.5*step_size)
    ys.append(0)
    for i in range(len(x)):
        xs.append(x[i] - 0.5*step_size)
        xs.append(x[i] + 0.5*step_size)
        ys.append(y[i])
        ys.append(y[i])
    xs.append(x[-1] + 0.5*step_size)
    ys.append(0)
    plt.plot(xs, ys, color=color, linewidth=linewidth)

In [ ]:
cmap = plt.get_cmap('Blues')
plt.figure(figsize=(1.5*1.18,1.3*1.18), dpi=400)
# for i, thresh in enumerate([0, 0.2, 0.4, 0.7 ]):
#     pass
    
threshs = [0, 0.2, 0.4, 0.7 , 1]
for i, thresh in enumerate(threshs[:-1]):
    histogram = np.histogram(all_uniqueness_scores[(all_uniqueness_scores['num_peaks']==1) &\
                                                   (all_uniqueness_scores['norm_ws_dist']>thresh)&\
                                                   (all_uniqueness_scores['norm_ws_dist']<=threshs[i+1])
                                                  ]['lambda_max'].values
                             
                             , bins=40, range=(100,900))
    plot_outline(histogram[1][:-1], histogram[0], 20, color=cmap(thresh+0.3), linewidth=1)

plt.yscale('log')
plt.ylabel('Number of molecules')
plt.xlabel('Wavelength (nm)')
plt.minorticks_on()
plt.savefig(f'{FIGURE_DIR}/{date_str}_distribution_of_single_peak_lmax_uniqueness_overlain.pdf', dpi=300)
plt.show()


## Look at most unique molecules

In [ ]:
for n in uniqueness_scores.sort_values('ws_dist', ascending=False).head(4)['name']:
    plt.figure(figsize=(1.5,1.1))
    plt.plot(pred_spec.loc[:, n], color='black')
    plt.xticks([0,200,400,600,800,1000])
    plt.xlim(0,1000)
    print (f'{FIGURE_DIR}/{date_str}_{n}_predicted_spectrum.pdf')
    plt.savefig(f'{FIGURE_DIR}/{date_str}_{n}_predicted_spectrum.pdf')
    plt.show()
    
    
print('.'.join(uniqueness_scores.sort_values('ws_dist', ascending=False).head(4)['smiles'].tolist()))
print(uniqueness_scores.sort_values('ws_dist', ascending=False).head(4)['norm_ws_dist'].tolist())